# Question 1: Option pricing using a binomial lattice

We price European and American puts and calls on the no-arbitrage lattice (Section 3.2 of the notes):

$$u = e^{\sigma\sqrt{\Delta t} + (r - \sigma^2/2)\Delta t}, \qquad d = e^{-\sigma\sqrt{\Delta t} + (r - \sigma^2/2)\Delta t}, \qquad q = \frac{e^{r\Delta t} - d}{u - d}.$$

At timestep $n$ the stock prices are $S^n_j = S^0_0\, u^j d^{\,n-j}$ for $j = 0, \dots, n$. Working backwards from the payoff at $n = N$,

$$V^n_j = e^{-r\Delta t}\left(q\, V^{n+1}_{j+1} + (1-q)\, V^{n+1}_j\right),$$

and in the American case $V^n_j \leftarrow \max\left(V^n_j,\ \text{payoff}(S^n_j)\right)$.

The code has one `for` loop over the timesteps. Inside it, all the work is done with vector operations. Only one column of the lattice is kept at a time, so storage is $O(N)$.

In [1]:
import numpy as np

try:
    from blsprice import blsprice  # the version provided with the assignment
except ImportError:
    from scipy.stats import norm

    # Fallback copy of blsprice with the same call as the Matlab function:
    # [Call, Put] = blsprice(Price, Strike, Rate, Time, Volatility, Yield)
    def blsprice(Price, Strike, Rate, Time, Volatility, Yield=0):
        d1 = (np.log(Price/Strike) + (Rate - Yield + Volatility**2/2)*Time) / (Volatility*np.sqrt(Time))
        d2 = d1 - Volatility*np.sqrt(Time)
        Call = Price*np.exp(-Yield*Time)*norm.cdf(d1) - Strike*np.exp(-Rate*Time)*norm.cdf(d2)
        Put = Strike*np.exp(-Rate*Time)*norm.cdf(-d2) - Price*np.exp(-Yield*Time)*norm.cdf(-d1)
        return Call, Put

In [2]:
# Parameters & Given Info (Table 1)
sigma = 0.32
r = 0.04
T = 1.0
K = 100
S0 = 100
N_list = [500, 1000, 2000, 4000, 8000]

In [3]:
def payoff(S, K, opttype):
    if opttype == 'call':
        return np.maximum(S - K, 0)
    else:
        return np.maximum(K - S, 0)


def binomial_price(S0, K, T, r, sigma, N, opttype='put', exercise='european'):
    """Price an option on the no-arbitrage binomial lattice.
    opttype: 'call' or 'put'; exercise: 'european' or 'american'."""
    dt = T/N
    u = np.exp(sigma*np.sqrt(dt) + (r - sigma**2/2)*dt)
    d = np.exp(-sigma*np.sqrt(dt) + (r - sigma**2/2)*dt)
    q = (np.exp(r*dt) - d)/(u - d)
    disc = np.exp(-r*dt)

    # Stock prices and option values at expiry (timestep N): S_j = S0 u^j d^(N-j), j = 0..N
    j = np.arange(N + 1)
    S = S0 * u**j * d**(N - j)
    V = payoff(S, K, opttype)

    # Step backwards in time, keeping only one column of the lattice
    for n in range(N - 1, -1, -1):
        V = disc*(q*V[1:] + (1 - q)*V[:-1])  # column n has n+1 nodes
        if exercise == 'american':
            S = S0 * u**j[:n+1] * d**(n - j[:n+1])  # stock prices in column n
            V = np.maximum(V, payoff(S, K, opttype))
    return V[0]

In [4]:
def convergence_table(opttype, exercise, exact=None):
    print(f'{exercise.capitalize()} {opttype}')
    print(f'{"dt":>8} {"Value":>14} {"Change":>14} {"Ratio":>10}')
    values = []
    for N in N_list:
        values.append(binomial_price(S0, K, T, r, sigma, N, opttype, exercise))
        line = f'{"T/" + str(N):>8} {values[-1]:14.8f}'
        if len(values) >= 2:
            line += f' {values[-1] - values[-2]:14.8f}'
        if len(values) >= 3:
            line += f' {(values[-2] - values[-3])/(values[-1] - values[-2]):10.4f}'
        print(line)
    if exact is not None:
        print(f'{"exact":>8} {exact:14.8f}')
    print()
    return values

In [5]:
call_exact, put_exact = blsprice(S0, K, r, T, sigma)

euro_call = convergence_table('call', 'european', call_exact)
euro_put = convergence_table('put', 'european', put_exact)

European call
      dt          Value         Change      Ratio
   T/500    14.52492129
  T/1000    14.52247140    -0.00244989
  T/2000    14.52008775    -0.00238365     1.0278
  T/4000    14.51938526    -0.00070248     3.3932
  T/8000    14.51995072     0.00056546    -1.2423
   exact    14.51959545

European put
      dt          Value         Change      Ratio
   T/500    10.60386520
  T/1000    10.60141531    -0.00244989
  T/2000    10.59903166    -0.00238365     1.0278
  T/4000    10.59832918    -0.00070248     3.3932


  T/8000    10.59889464     0.00056546    -1.2423
   exact    10.59853937



In [6]:
# Errors against the exact solution, to see how fast they shrink
print(f'{"dt":>8} {"call error":>14} {"put error":>14}')
for N, c, p in zip(N_list, euro_call, euro_put):
    print(f'{"T/" + str(N):>8} {c - call_exact:14.2e} {p - put_exact:14.2e}')

      dt     call error      put error
   T/500       5.33e-03       5.33e-03
  T/1000       2.88e-03       2.88e-03
  T/2000       4.92e-04       4.92e-04
  T/4000      -2.10e-04      -2.10e-04
  T/8000       3.55e-04       3.55e-04


## Estimating the limit of the ratio (by hand)

From equation (1), at $(S^0_0, 0)$ write $V_{\Delta t} = V^{exact} + C\Delta t + O((\Delta t)^2)$. Then

$$V_{\Delta t/2} - V_{\Delta t} = C\left(\tfrac{\Delta t}{2} - \Delta t\right) + O((\Delta t)^2) = -\tfrac{C\Delta t}{2} + O((\Delta t)^2),$$

$$V_{\Delta t/4} - V_{\Delta t/2} = C\left(\tfrac{\Delta t}{4} - \tfrac{\Delta t}{2}\right) + O((\Delta t)^2) = -\tfrac{C\Delta t}{4} + O((\Delta t)^2).$$

So

$$\lim_{\Delta t \to 0} \frac{V_{\Delta t/2} - V_{\Delta t}}{V_{\Delta t/4} - V_{\Delta t/2}} = \frac{-C\Delta t/2}{-C\Delta t/4} = 2.$$

In general, if the error is $O((\Delta t)^p)$, the ratio tends to $2^p$. A ratio near 2 means first order, and a ratio near 4 means second order.

## Observations (European options)

Both tables converge to the `blsprice` values (call $\approx 14.5196$, put $\approx 10.5985$). The error drops from about $5\times10^{-3}$ at $\Delta t = T/500$ to below $5\times10^{-4}$ at $T/8000$. That is roughly 16 times smaller for 16 times as many steps, so the error is shrinking roughly like $\Delta t$. This matches first-order accuracy.

The Ratio column does **not** settle down at 2, though. It jumps around (values such as 1.0, 3.4, and even negative numbers), and the Change column changes sign. The cause is the kinked payoff $\max(S-K,0)$ at $S = K$. On this lattice the strike does not sit at the same place relative to the nodes for each $N$. So each time $N$ doubles, the error picks up an extra term that oscillates with $N$, on top of the smooth $C\Delta t$ term. This breaks the clean expansion (1), which assumes a smooth payoff. The overall size of the error still falls like $\Delta t$, so the method is first order, in line with the theory. But we cannot get the clean ratio of 2 without smoothing the payoff, which this question tells us not to do.

The call and put tables have the same Change and Ratio columns. This is put-call parity: on this lattice the discounted stock price is a martingale, so $C - P = S^0_0 - Ke^{-rT}$ holds exactly for every $N$. Any error in the call is matched by the same error in the put.

In [7]:
amer_put = convergence_table('put', 'american')

American put
      dt          Value         Change      Ratio
   T/500    11.00077610
  T/1000    10.99799340    -0.00278270
  T/2000    10.99577049    -0.00222292     1.2518


  T/4000    10.99513614    -0.00063435     3.5043


  T/8000    10.99545197     0.00031583    -2.0085



## Observations (American put)

The American put converges to about $10.995$. This is higher than the European put ($10.5985$), because the right to exercise early has value for a put. The Change column again gets smaller as $\Delta t$ shrinks, but the Ratio column is erratic, just as in the European case. The same kink in the payoff at $S = K$ is to blame. The early exercise boundary adds another source of error, because it also falls between lattice nodes. The changes shrink roughly in proportion to $\Delta t$ overall, which again points to first-order convergence.